# Chapter 27 — The Session File Format

**Companion to *Pi Agents*** · [`Pi Agents` chapter 27](https://programmer.ie/books/pi/27-chapter/)

| | |
|---|---|
| Chapter | `27-chapter.md` · weight 27 · `/books/pi/27-chapter/` |
| Notebook | `notebooks/pi/27-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Is it really JSONL with a header first, and are there two timestamp formats on
one field name?

The chapter's point: the session file is JSONL — a header, then one entry per
line, linked by `id` and `parentId`. An entry carries an **ISO** timestamp; the
message inside it carries a **Unix milliseconds** number. Both are on the field
name `timestamp`, so the type tells you which one you have.


## What this notebook establishes

- The file is **JSONL**: a session header first, then one entry per line, each linked by `id` and `parentId`.
- The header has **type `session`, version 3**, and **no `parentId`** — it is not part of the tree.
- Every entry has the base fields `id`, `parentId` and `timestamp`.
- For a linear conversation, each entry's `parentId` is the entry before it — the tree is a chain when nothing branches.
- **Two timestamp formats on one field name**: the entry's `timestamp` is ISO (`2026-…T…`), the message's `timestamp` is a Unix-milliseconds number.
- The chapter's own **reader runs against a real session file** and prints the session, the messages and the model.

## What this notebook does **not** establish

- **The header `version` moves** — 3 at this pin; the format is readable, not guaranteed across releases.
- **The projection passes and the per-entry-type visibility table** have no runtime test here — chapter 5 covers visibility.
- **The declaration comparison** (`assert-equal.ts`) is a compile-time check, not a runtime one.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** + **Inspect** — a real persisted session file parsed directly, plus the chapter's own reader run as a child process and the declaration check.
* **Evidence scope:** `in_process_runtime` + `declarations`. The format is Pi's; the read is direct.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(27))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/27-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: a real session file

In [3]:
SESS = pinb.driver_source("ch27-session-entries.ts")
sess = pinb.driver(SESS, label="ch27-session-entries", timeout=180)
print(pinb.md_table(
    ["field", "value"],
    [["header type", sess["header"]["type"]],
     ["header version", sess["header"]["version"]],
     ["header has parentId", "yes" if sess["header"]["hasParentId"] else "no"],
     ["entries", sess["entries"]["count"]],
     ["entry types", ", ".join(sess["entries"]["types"])],
     ["all base fields present", "yes" if sess["entries"]["baseFieldsOk"] else "no"],
     ["linear chain", "yes" if sess["entries"]["linearChain"] else "no"]],
))

| field | value |
|---|---|
| header type | session |
| header version | 3 |
| header has parentId | no |
| entries | 5 |
| entry types | model_change, thinking_level_change, message, message, message |
| all base fields present | yes |
| linear chain | yes |


In [4]:
pinb.show_checks([
    pinb.check("the first record is the session header",
               sess["header"]["type"] == "session", "type session"),
    pinb.check("the header has no parentId (not part of the tree)",
               not sess["header"]["hasParentId"], "no parentId"),
    pinb.check("every entry has id, parentId and timestamp",
               sess["entries"]["baseFieldsOk"], "base fields present"),
    pinb.check("the entries form a chain for a linear conversation",
               sess["entries"]["linearChain"], "each parent is the entry before"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------  -------------------------------
PASS  the first record is the session header  type session
PASS  the header has no parentId (not part of the tree)  no parentId
PASS  every entry has id, parentId and timestamp  base fields present
PASS  the entries form a chain for a linear conversation  each parent is the entry before

4/4 assertions held.


## Two timestamps, two formats on one field name

In [5]:
print(pinb.md_table(
    ["where", "format", "present?"],
    [["entry.timestamp", "ISO string", "yes" if sess["timestamps"]["isoOnEntry"] else "no"],
     ["entry.message.timestamp", "Unix milliseconds", "yes" if sess["timestamps"]["unixOnMessage"] else "no"]],
))

| where | format | present? |
|---|---|---|
| entry.timestamp | ISO string | yes |
| entry.message.timestamp | Unix milliseconds | yes |


In [6]:
pinb.show_checks([
    pinb.check("the entry timestamp is ISO",
               sess["timestamps"]["isoOnEntry"], "ISO string"),
    pinb.check("the message timestamp is a Unix-milliseconds number",
               sess["timestamps"]["unixOnMessage"], "number"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------------  ----------
PASS  the entry timestamp is ISO  ISO string
PASS  the message timestamp is a Unix-milliseconds number  number

2/2 assertions held.


## The chapter's reader, run against a real file

`ch27-session-entries/read-session.ts` is the reading program the chapter prints.
Run as a child process against the session written above, it is the chapter's own
reader against the chapter's own file.

In [7]:
print("Reader stdout:")
print(sess["reader"]["stdout"])
print()
pinb.show_checks([
    pinb.check("the reader exits 0", sess["reader"]["status"] == 0, "exit 0"),
    pinb.check("it prints the session version", sess["reader"]["hasSessionLine"], "Session v3"),
    pinb.check("it prints the user message", sess["reader"]["hasUser"], "user: hello"),
    pinb.check("it prints the assistant message", sess["reader"]["hasAssistant"], "assistant: hello back"),
])

Reader stdout:
Session v3: 01a11844-c335-773b-ad2e-97163972c49c
[545cc96d] Model: faux/faux-1
[7ad0545b] Thinking: off
[ce8b8900] system: ""
[e8c611cc] user: [{"type":"text","text":"hello"}]
[ba9c1731] assistant: [{"type":"text","text":"hello back"}]


== Assertions ==
  assertion  observed
----  -------------------------------  ---------------------
PASS  the reader exits 0  exit 0
PASS  it prints the session version  Session v3
PASS  it prints the user message  user: hello
PASS  it prints the assistant message  assistant: hello back

4/4 assertions held.


## The chapter's own tests

All three entry tests, plus the declaration comparison run by `tsc`.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [8]:
PATTERNS = 'ch27-session-entries/entries.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch27-session-entries/entries.test.ts', show="timestamp")

$ node --test ch27-session-entries/entries.test.ts
  PASS  3 passed, 0 failed, 0 skipped  in 1006 ms

  tests matching 'timestamp':
    ok   two timestamps, two formats: ISO on the entry, Unix milliseconds on the message


## Your turn: predict, then run

**Predict first.** The header's `version` is 3 at this pin. What happens if you
write a file with `version: 1`? Does Pi read it or refuse it?

**Then change one input.** `read-session.ts` switches on `entry.type`. Add a
`context_edit` row to the reader. What does a `context_edit` entry look like in
the file? (Chapter 5 has the shape.)

**Predict the boundary.** The driver reads a linear session. What does the entry
chain look like after a `/tree` branch — is `parentId` still the entry before?

In [9]:
print("Predictions:")
print("  1. version 1: Pi reads it (the version is informational at read time).")
print("  2. context_edit entry: has targetId and replacement; chapter 5 has the shape.")
print("  3. After a branch: parentId points to the branch point, not the previous line.")
print()
print("Observed above:")
print(f"  header: type={sess['header']['type']}, version={sess['header']['version']}, parentId present={sess['header']['hasParentId']}")
entry_types = ", ".join(sess["entries"]["types"])
print(f"  entries: {sess['entries']['count']} ({entry_types})")
print(f"  timestamps: ISO entry={sess['timestamps']['isoOnEntry']}, Unix message={sess['timestamps']['unixOnMessage']}")
print(f"  reader: exit={sess['reader']['status']}")
print()
assert sess["header"]["type"] == "session"
assert not sess["header"]["hasParentId"]
assert sess["entries"]["baseFieldsOk"] and sess["entries"]["linearChain"]
assert sess["timestamps"]["isoOnEntry"] and sess["timestamps"]["unixOnMessage"]
assert sess["reader"]["status"] == 0
print("held: JSONL with a header; entry chain; two timestamp formats on one field name")

Predictions:
  1. version 1: Pi reads it (the version is informational at read time).
  2. context_edit entry: has targetId and replacement; chapter 5 has the shape.
  3. After a branch: parentId points to the branch point, not the previous line.

Observed above:
  header: type=session, version=3, parentId present=False
  entries: 5 (model_change, thinking_level_change, message, message, message)
  timestamps: ISO entry=True, Unix message=True
  reader: exit=0

held: JSONL with a header; entry chain; two timestamp formats on one field name


## Interpretation

The session file is a **tree serialised as JSONL**, and each field has a job:

| Field | Type | What it tells you |
|---|---|---|
| `type` (header) | `"session"` | This line is the header, not an entry |
| `version` (header) | number | The format version — 3 at this pin |
| `id` | string | The entry's identity, used as a durable cursor |
| `parentId` | string \| null | The parent in the tree; `null` for a root |
| `timestamp` (entry) | ISO string | When the entry was appended |
| `message.timestamp` | Unix milliseconds | When the message was created |

The practical rules:
1. **Read JSONL line by line** — one JSON object per line, LF-terminated.
2. **The header is not an entry** — skip it when walking the tree.
3. **`id` is a durable cursor** — pass it as `since` to `get_entries` for incremental reads.
4. **Check the type of `timestamp`** — ISO on the entry, number on the message.
5. **A branch breaks the chain** — `parentId` points at the branch point, not the previous line.

## Sources, execution mode and limitations

**Execution mode:** **Run** + **Inspect** — a real persisted session file parsed directly, plus the chapter's own reader run as a child process and the declaration check.

**Evidence scope:** `in_process_runtime` + `declarations`. The format is Pi's; the read is direct.

### What was read or run

- **Ran** `drivers/ch27-session-entries.ts`: a real persisted session written, parsed directly, and read by the chapter's own `read-session.ts` as a child process.
- **Ran** `ch27-session-entries/entries.test.ts` (3 tests).
- **Ran** `tsc` over `ch27-session-entries/assert-equal.ts` (the declaration check).
- **Read** `examples/evidence.json`, chapter 27 rows (4 claims).

### Limitations

- **The header `version` moves** — 3 at this pin; the format is readable, not guaranteed across releases.
- **The projection passes and the per-entry-type visibility table** have no runtime test here — chapter 5 covers visibility.
- **The declaration comparison** is a compile-time check, not a runtime one.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
